In [1]:
import cv2
from ultralytics import YOLO
import numpy as np

model=YOLO('yolov8n.pt')

In [2]:
video_path= 'Sample_Video_HighQuality.mp4'

In [3]:
cap=cv2.VideoCapture(video_path)
if not cap.isOpened():
    print(f"Error:couldn't open video file'{video_path}'")
    exit()

In [4]:
fps=int(cap.get(cv2.CAP_PROP_FPS))
width=int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height=int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
print(f"video:{width}x{height}@ {fps}fps")

video:1920x1080@ 24fps


In [5]:
fourcc=cv2.VideoWriter_fourcc(*'mp4v')
out=cv2.VideoWriter('output_video.mp4',fourcc,fps,(width,height))

In [6]:
import numpy as np
frame_number=0
while cap.isOpened():
    ret,frame=cap.read()
    if not ret:
        break
    frame_number +=1
    results=model(frame,conf=0.5,verbose=False)
    results=results[0]
    cars=[]
    people=[]
    for box in results.boxes:
        cls=int(box.cls[0])
        conf=float(box.conf[0])
        x1,y1,x2,y2=map(int,box.xyxy[0])
        if cls==2:
            cars.append((x1,y1,x2,y2))
        elif cls==0:
            people.append((x1,y1,x2,y2))
    blue_cars = 0
    other_cars = 0
    
    for i, (x1, y1, x2, y2) in enumerate(cars):
        car_crop = frame[y1:y2, x1:x2]
        
        if car_crop.size == 0:
            continue
        
        # Convert to HSV
        hsv = cv2.cvtColor(car_crop, cv2.COLOR_BGR2HSV)
        
        # Blue range in HSV
        lower_blue = np.array([90, 50, 50])
        upper_blue = np.array([130, 255, 255])
        
        mask = cv2.inRange(hsv, lower_blue, upper_blue)
        blue_ratio = np.sum(mask > 0) / (mask.size)
        
        # Red box for blue cars, blue box for others
        if blue_ratio > 0.2:  # 20%+ blue = blue car
            color_box = (0, 0, 255)  # Red BGR
            label = f'Blue Car'
            blue_cars += 1
        else:
            color_box = (255, 0, 0)  # Blue BGR
            label = f'Car'
            other_cars += 1
        
        # Draw box + label
        cv2.rectangle(frame, (x1, y1), (x2, y2), color_box, 3)
        cv2.putText(frame, label, (x1, y1-10), 
                   cv2.FONT_HERSHEY_SIMPLEX, 0.7, color_box, 2)
    
    # Step 7: Draw people (green boxes)
    for x1, y1, x2, y2 in people:
        cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
        cv2.putText(frame, 'Person', (x1, y1-10), 
                   cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
    
    # Step 8: Show COUNTS
    # Create background for text
    cv2.rectangle(frame, (0, 0), (400, 120), (0, 0, 0), -1)
    
    # Display counts for THIS FRAME
    cv2.putText(frame, f'Cars: {len(cars)}', (10, 30), 
               cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)
    cv2.putText(frame, f'Blue Cars: {blue_cars}', (10, 60), 
               cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 2)
    cv2.putText(frame, f'Other Cars: {other_cars}', (10, 90), 
               cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 0, 0), 2)
    cv2.putText(frame, f'People: {len(people)}', (10, 120), 
               cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)
    
    # Show frame number
    cv2.putText(frame, f'Frame: {frame_number}', (width-200, 30), 
               cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 0), 2)
    
    # Step 9: Display & Save
    cv2.imshow('Traffic Detector', frame)
    out.write(frame)
    
    # Print progress every 30 frames
    if frame_number % 30 == 0:
        print(f"Processed frame {frame_number}")
    
    # Controls
    key = cv2.waitKey(1) & 0xFF
    if key == ord('q'):
        print("Quitting...")
        break
    elif key == ord('s'):
        cv2.imwrite(f'snapshot_frame_{frame_number}.jpg', frame)
        print(f"Saved snapshot_frame_{frame_number}.jpg!")

# Cleanup
cap.release()
out.release()
cv2.destroyAllWindows()

print("\n" + "="*50)
print("PROCESSING COMPLETE!")
print("="*50)
print(f"Total frames processed: {frame_number}")
print(f"Output saved to: output_video.mp4")
print("="*50)



Processed frame 30
Processed frame 60
Processed frame 90
Processed frame 120
Processed frame 150
Processed frame 180
Processed frame 210
Processed frame 240
Processed frame 270
Processed frame 300
Processed frame 330
Processed frame 360
Processed frame 390
Processed frame 420
Processed frame 450
Processed frame 480
Processed frame 510
Processed frame 540
Processed frame 570
Processed frame 600
Processed frame 630
Processed frame 660
Processed frame 690
Processed frame 720
Processed frame 750
Processed frame 780
Processed frame 810
Processed frame 840
Processed frame 870
Processed frame 900
Processed frame 930
Processed frame 960
Processed frame 990

PROCESSING COMPLETE!
Total frames processed: 1001
Output saved to: output_video.mp4


In [ ]:

image_path = 'image1.png'  # ← CHANGE THIS to your image name

frame = cv2.imread(image_path)

if frame is None:
    print(f"ERROR: Could not find '{image_path}'")
else:
    # Lower confidence to catch people + use larger model for accuracy
    results = model(frame, conf=0.35, verbose=False)  # lowered from 0.5
    result = results[0]

    blue_cars = 0
    other_cars = 0
    people = 0

    for box in result.boxes:
        cls = int(box.cls[0])
        conf = float(box.conf[0])
        x1, y1, x2, y2 = map(int, box.xyxy[0])

        if cls == 2:  # Car
            car_crop = frame[y1:y2, x1:x2]
            if car_crop.size == 0:
                continue

            hsv = cv2.cvtColor(car_crop, cv2.COLOR_BGR2HSV)

            # Mask 1 - Bright/normal blue
            lower_blue1 = np.array([100, 80, 50])
            upper_blue1 = np.array([125, 255, 255])
            mask1 = cv2.inRange(hsv, lower_blue1, upper_blue1)
            #Mask 2 - Dark blue only (high hue, low brightness, decent saturation)
            lower_blue2 = np.array([100, 60, 20])
            upper_blue2 = np.array([130, 255, 80])   # ← brightness capped LOW (dark cars only)
            mask2 = cv2.inRange(hsv, lower_blue2, upper_blue2)

            # Combine both masks
            mask = cv2.bitwise_or(mask1, mask2)
            h, w = mask.shape
            center_mask = mask[h//4:3*h//4, w//4:3*w//4]
            blue_ratio = np.sum(center_mask > 0) / center_mask.size
            if blue_ratio > 0.25: # raised from 0.2 to reduce false positives
                cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 0, 255), 3)
                cv2.putText(frame, f'Blue Car {conf:.0%}', (x1, y1-10),
                           cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 255), 2)
                blue_cars += 1
            else:
                cv2.rectangle(frame, (x1, y1), (x2, y2), (255, 0, 0), 3)
                cv2.putText(frame, f'Car {conf:.0%}', (x1, y1-10),
                           cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 0, 0), 2)
                other_cars += 1

        elif cls == 0:  # ✅ Person
            cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
            cv2.putText(frame, f'Person {conf:.0%}', (x1, y1-10),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
            people += 1

        elif cls == 3:  # ✅ Motorcycle
            cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 255), 2)
            cv2.putText(frame, f'Motorcycle {conf:.0%}', (x1, y1-10),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 255), 2)

    # Count display
    cv2.rectangle(frame, (0, 0), (400, 130), (0, 0, 0), -1)
    cv2.putText(frame, f'Cars: {other_cars}',     (10, 30),  cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)
    cv2.putText(frame, f'Blue Cars: {blue_cars}', (10, 60),  cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 2)
    cv2.putText(frame, f'People: {people}',       (10, 90),  cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)

    cv2.imwrite('test_output.jpg', frame)
    print(f"✅ Blue Cars: {blue_cars}")
    print(f"🚗 Other Cars: {other_cars}")
    print(f"🚶 People: {people}")
    print("Saved as test_output.jpg!")

    cv2.imshow('Test Result', frame)
    cv2.waitKey(0)
    cv2.destroyAllWindows()


✅ Blue Cars: 14
🚗 Other Cars: 20
🚶 People: 0
Saved as test_output.jpg!
